# Building and Testing a RESTful API with FastAPI

In this notebook, we build and test a high-performance **REST API** using **FastAPI** and **Pydantic**. We demonstrate API routing, request validation, data serialization, and in-memory CRUD operations tested with `TestClient`.


### Importing the Dependencies


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional, List
import pandas as pd


### Defining Data Schemas (Pydantic Models)


In [ ]:
class TaskCreate(BaseModel):
    title: str
    description: Optional[str] = None
    priority: str = "Medium"

class TaskResponse(BaseModel):
    id: int
    title: str
    description: Optional[str] = None
    priority: str
    completed: bool = False


### Initializing FastAPI Application & Endpoints


In [ ]:
app = FastAPI(title="Task Management API", version="1.0")

# In-memory database
tasks_db = []
id_counter = 1

@app.get("/")
def root():
    return {"message": "FastAPI Task Server is Running Live!", "status": "Healthy"}

@app.get("/tasks", response_model=List[TaskResponse])
def get_tasks():
    return tasks_db

@app.post("/tasks", response_model=TaskResponse, status_code=201)
def create_task(task: TaskCreate):
    global id_counter
    new_task = TaskResponse(
        id=id_counter,
        title=task.title,
        description=task.description,
        priority=task.priority,
        completed=False
    )
    tasks_db.append(new_task)
    id_counter += 1
    return new_task

@app.put("/tasks/{task_id}/complete", response_model=TaskResponse)
def complete_task(task_id: int):
    for task in tasks_db:
        if task.id == task_id:
            task.completed = True
            return task
    raise HTTPException(status_code=404, detail="Task not found")


### Testing Endpoints Interactively with TestClient


In [ ]:
client = TestClient(app)

# 1. Health check
res_health = client.get("/")
print("Health Check:", res_health.json())

# 2. Create tasks
task1 = client.post("/tasks", json={"title": "Complete Machine Learning Coursework", "priority": "High"})
task2 = client.post("/tasks", json={"title": "Submit GCI World 2026 Prediction", "priority": "Critical"})
task3 = client.post("/tasks", json={"title": "Deploy AI agency pipeline in n8n", "priority": "High"})

print("\nCreated Task 1:", task1.json())
print("Created Task 2:", task2.json())


### Retrieving All Tasks & Viewing Formatted Table


In [ ]:
res_all = client.get("/tasks")
df_api = pd.DataFrame(res_all.json())
df_api


### Updating Task Status (Marking Task #2 as Done)


In [ ]:
res_update = client.put("/tasks/2/complete")
print("Updated Task:", res_update.json())

# Re-fetch all tasks to verify update
pd.DataFrame(client.get("/tasks").json())
